# T/NK Cell ONLY Merged Pipeline v2.0
**Architecture v2.0 — CellTypist-equal scANVI:**
1. Ref + Query concatenated **symmetrically**
2. CellTypist annotates **all cells** (ref + query together)
3. scVI batch correction on HVG
4. scANVI trains on **CellTypist labels for all cells** — query is NOT forced to Unknown
5. Low-confidence CellTypist cells (< threshold) → `Unknown` for scANVI
6. Reference original labels preserved as `cell_type_fine_ref` (QC only, not used for training)

## Cell 0 — Imports & Global Settings

In [1]:
import os
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"]  = "1"

import sys, warnings, json, gc, joblib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.sparse import issparse, csr_matrix
from scipy.stats import entropy

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import scanpy as sc
import scvi
import celltypist
from celltypist import models
from umap import UMAP

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

print("=" * 80)
print("T/NK Cell ONLY Merged Pipeline v2.0  (CellTypist-equal scANVI)")
print("=" * 80)
gpu_available = torch.cuda.is_available()
print(f"GPU available: {gpu_available}")
if gpu_available:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
scvi.settings.dl_num_workers = 0
import time
import traceback


/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


T/NK Cell ONLY Merged Pipeline v2.0  (CellTypist-equal scANVI)
GPU available: True
GPU: Tesla V100-SXM2-16GB


## Cell 1 — Configuration

In [2]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================

REFERENCE_H5AD = "/home/h2048/data/py/0129/tnk_analysis_unified/results/subcluster_unified_v2_20260129/adata_tnk_subclustered_FINAL_v2_0_1_20260129.h5ad"
QUERY_H5AD     = "/home/h2048/data/py/0127/scarches_mapping_FIXED_v1_2/subsets/t_cells.h5ad"

# Reference label column — kept as QC comparison only, NOT used for scANVI training
REF_LABEL_FINE = "cell_type_L3"

BATCH_KEY  = "sample"
TISSUE_KEY = "tissue"

OUTPUT_DIR    = "/home/h2048/data/py/20260310/tcell_only_merged_pipeline_v2"
OUTPUT_PREFIX = "tcell_merged_v2"

INCLUDE_COARSE_TYPES = None

N_HVG                = 4000
FORCE_MARKERS_IN_HVG = True

SCVI_N_LATENT   = 100
SCVI_N_LAYERS   = 2
SCVI_N_HIDDEN   = 128
SCVI_DROPOUT    = 0.1
MAX_EPOCHS_SCVI = 400

MAX_EPOCHS_SCANVI  = 200
UNLABELED_CATEGORY = "Unknown"

BATCH_SIZE    = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 0.0

# CellTypist — annotates ALL merged cells (ref + query equally)
CELLTYPIST_MODEL         = "/home/h2048/data/source/reference/celltypist_models/Immune_All_Low.pkl"
CELLTYPIST_MAJORITY_VOTE = True

CELLTYPIST_DIRECT_LABEL_KEY = "celltypist_label_direct"
CELLTYPIST_DIRECT_FILT_KEY  = "celltypist_label_direct_filt"
CELLTYPIST_CONF_THRESHOLD   = 0.5   # below this -> Unknown for scANVI
CELLTYPIST_SAVE_PROBA       = True

# scANVI label source:
#   "direct"   -> celltypist_label_direct       (unfiltered)
#   "filtered" -> celltypist_label_direct_filt  (low-conf -> Unknown)
SCANVI_LABEL_SOURCE = "filtered"

QUERY_LEIDEN_RESOLUTION = 1.0

# CD4/CD8 signature (kept for QC visualization only)
CD4_MARKERS            = ["CD4","IL7R","FOXP3","IL2RA","CTLA4","CCR7","SELL"]
CD8_MARKERS            = ["CD8A","CD8B","GZMB","PRF1","IFNG","NKG7","GNLY"]
CD4_SCORE_THRESHOLD    = 0.3
CD8_SCORE_THRESHOLD    = 0.3
NK_MARKERS             = ["GNLY","NKG7","KLRD1","NCAM1","FCGR3A","TYROBP"]
TREG_MARKERS           = ["FOXP3","IL2RA","CTLA4","IKZF2","TNFRSF18"]
TRM_MARKERS            = ["ITGAE","CXCR6","CD69","ITGA1","ZNF683"]
EXHAUSTION_MARKERS     = ["HAVCR2","TIGIT","LAG3","PDCD1","TOX","NR4A1"]
NAIVE_MARKERS          = ["CCR7","SELL","TCF7","LEF1","CD27","CD28"]
EFFECTOR_MARKERS       = ["GZMB","PRF1","IFNG","TNF","CX3CR1","FGFBP2"]
PROLIF_MARKERS         = ["MKI67","TOP2A","PCNA"]

FORCED_MARKERS = list(set(
    CD4_MARKERS + CD8_MARKERS + NK_MARKERS + TREG_MARKERS + TRM_MARKERS +
    EXHAUSTION_MARKERS + NAIVE_MARKERS + EFFECTOR_MARKERS + PROLIF_MARKERS +
    ["CD3D","CD3E","CD3G","TRAC","TRBC1","TRBC2","CD56","CD16"]
))

STRESS_SIGNATURE_GENES = [
    "HSPA1A","HSPA1B","HSPA8","HSP90AA1","HSP90AB1","DNAJB1",
    "JUN","JUNB","JUND","FOS","FOSB","EGR1","IER2"
]
S_GENES = [
    "MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UHRF1",
    "GINS2","MCM6","CDCA7","DTL","PRIM1","HELLS","RFC2","RPA2",
    "NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7",
    "POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1"
]
G2M_GENES = [
    "HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80",
    "CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","FAM64A",
    "SMC4","CCNB1","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E"
]

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sc.settings.seed   = RANDOM_SEED
scvi.settings.seed = RANDOM_SEED

# Visualization & timing constants
PIPELINE_START = time.time()
FIGURE_DPI    = 300
FIGURE_FORMAT = "pdf"
UMAP_SIZE     = 3
UMAP_ALPHA    = 0.6


Seed set to 42


## Cell 2 — Helper Functions

In [3]:
# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================

def ensure_counts_layer(adata, counts_layer="counts"):
    if counts_layer not in (adata.layers or {}):
        print(f"  [WARN] layers['{counts_layer}'] not found, checking .X...")
        if hasattr(adata, 'X') and adata.X is not None:
            X_s = adata.X[:1000].toarray().flatten() if issparse(adata.X) else np.asarray(adata.X[:1000]).ravel()
            s   = np.asarray(X_s, dtype=np.float64)
            if np.any(s < 0):
                raise ValueError(".X contains negative values!")
            if np.allclose(s, np.round(s), atol=1e-6):
                print(f"  -> Auto-copying .X to layers['{counts_layer}']")
                adata.layers[counts_layer] = csr_matrix(adata.X) if issparse(adata.X) else adata.X.copy()
            else:
                raise ValueError(f"CRITICAL: .X not integer counts. Range: [{s.min():.4f},{s.max():.4f}]")
        else:
            raise ValueError(f"CRITICAL: layers['{counts_layer}'] missing and .X is None.")
    X_c = adata.layers[counts_layer]
    sd  = X_c.data[:1000] if issparse(X_c) else X_c.flat[:1000]
    s   = np.asarray(sd, dtype=np.float64)
    if s.size == 0:
        print(f"  [WARN] sampled counts empty; skipping integer check")
    else:
        if np.any(s < 0):
            raise ValueError("counts contains negative values!")
        if not np.allclose(s, np.round(s), atol=1e-6):
            raise ValueError(f"counts looks non-integer. Range: [{s.min():.4f},{s.max():.4f}]")
    if issparse(X_c) and not isinstance(X_c, csr_matrix):
        adata.layers[counts_layer] = csr_matrix(X_c)
    return counts_layer


def ensure_batch_tissue(adata, batch_key, tissue_key):
    for key, placeholder in [(batch_key, "unknown_batch"), (tissue_key, "unknown_tissue")]:
        if key not in adata.obs.columns:
            print(f"  [WARN] {key} not found, creating placeholder")
            adata.obs[key] = placeholder
        adata.obs[key] = adata.obs[key].astype("string").fillna(placeholder).astype("category")
        if placeholder not in adata.obs[key].cat.categories:
            adata.obs[key] = adata.obs[key].cat.add_categories([placeholder])


def compute_module_score_efficient(adata, gene_list, score_name, counts_layer="counts"):
    genes    = [g for g in gene_list if g in adata.var_names]
    gene_idx = adata.var_names.get_indexer(genes)
    gene_idx = gene_idx[gene_idx >= 0]
    if len(gene_idx) < 3:
        adata.obs[score_name] = 0.0
        adata.obs[f"{score_name}_norm"] = 0.0
        return
    adata_tmp = sc.AnnData(X=adata.layers[counts_layer][:, gene_idx].copy(),
                           var=adata.var.iloc[gene_idx].copy())
    sc.pp.normalize_total(adata_tmp, target_sum=1e4)
    sc.pp.log1p(adata_tmp)
    X_n = adata_tmp.X
    me  = np.asarray(X_n.mean(axis=1)).flatten() if issparse(X_n) else np.asarray(X_n).mean(axis=1).flatten()
    adata.obs[score_name] = me
    mn, mx = float(me.min()), float(me.max())
    adata.obs[f"{score_name}_norm"] = (me - mn) / (mx - mn) if mx > mn else 0.0
    del adata_tmp; gc.collect()


def compute_tnk_scores(adata):
    # QC-purpose scores only — NOT used for scANVI label assignment
    print("  -> Computing T/NK QC scores...")
    compute_module_score_efficient(adata, CD4_MARKERS,        "CD4_score")
    compute_module_score_efficient(adata, CD8_MARKERS,        "CD8_score")
    compute_module_score_efficient(adata, NK_MARKERS,         "NK_score")
    compute_module_score_efficient(adata, TREG_MARKERS,       "Treg_score")
    compute_module_score_efficient(adata, EXHAUSTION_MARKERS, "Exhaustion_score")
    compute_module_score_efficient(adata, NAIVE_MARKERS,      "Naive_score")
    compute_module_score_efficient(adata, TRM_MARKERS,        "TRM_score")
    cd4 = adata.obs["CD4_score_norm"] > CD4_SCORE_THRESHOLD
    cd8 = adata.obs["CD8_score_norm"] > CD8_SCORE_THRESHOLD
    adata.obs["cd4_cd8_by_score"] = np.select(
        [cd4 & ~cd8, cd8 & ~cd4, cd4 & cd8],
        ["CD4_single", "CD8_single", "DP"],
        default="DN"
    )
    for lbl in ["CD4_single","CD8_single","DP","DN"]:
        print(f"    {lbl}: {(adata.obs['cd4_cd8_by_score']==lbl).sum()}")


def prepare_covariates(adata):
    print("  -> Validating counts...")
    ensure_counts_layer(adata, "counts")
    print("  -> Batch/Tissue...")
    ensure_batch_tissue(adata, BATCH_KEY, TISSUE_KEY)
    print("  -> Signature scores...")
    if "pct_counts_mt" not in adata.obs.columns:
        adata.var["mt"] = adata.var_names.str.startswith("MT-")
        sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, layer="counts")
    compute_tnk_scores(adata)
    compute_module_score_efficient(adata, STRESS_SIGNATURE_GENES, "stress_score")
    if not all(k in adata.obs.columns for k in ["S_score","G2M_score"]):
        s_in = [g for g in S_GENES   if g in adata.var_names]
        g_in = [g for g in G2M_GENES if g in adata.var_names]
        if len(s_in) >= 5 and len(g_in) >= 5:
            cc  = list(dict.fromkeys(s_in + g_in))
            idx = adata.var_names.get_indexer(cc)
            ad_ = sc.AnnData(X=adata.layers["counts"][:, idx].copy(), var=adata.var.iloc[idx].copy())
            sc.pp.normalize_total(ad_, target_sum=1e4); sc.pp.log1p(ad_)
            sc.tl.score_genes_cell_cycle(ad_, s_genes=s_in, g2m_genes=g_in)
            adata.obs["S_score"]   = ad_.obs["S_score"].values
            adata.obs["G2M_score"] = ad_.obs["G2M_score"].values
            adata.obs["phase"]     = ad_.obs["phase"].values
            del ad_; gc.collect()
        else:
            adata.obs["S_score"] = 0.0; adata.obs["G2M_score"] = 0.0; adata.obs["phase"] = "G1"


def run_celltypist_on_full_genes(adata_merged, model_name=CELLTYPIST_MODEL, majority_vote=True):
    # Annotates ALL merged cells (ref + query) equally.
    # symbol_base matching handles var_names_make_unique suffixes.
    print("\n[CellTypist] Annotating ALL cells (ref + query)...")
    if os.path.isfile(model_name):
        model = models.Model.load(model_name)
        print(f"  -> Loaded: {model_name}")
    else:
        bn = os.path.basename(model_name)
        models.download_models(model=bn)
        model = models.Model.load(bn)

    model_genes = set(model.features)
    if "symbol_base" in adata_merged.var.columns:
        base_to_var = {}
        for vn, sb in zip(adata_merged.var_names, adata_merged.var["symbol_base"]):
            if sb in model_genes and sb not in base_to_var:
                base_to_var[sb] = vn
        available_genes = list(base_to_var.values())
        print(f"  -> {len(available_genes)}/{len(model_genes)} genes matched via symbol_base")
    else:
        available_genes = [g for g in adata_merged.var_names if g in model_genes]
        print(f"  -> {len(available_genes)}/{len(model_genes)} genes matched via var_names")

    if len(available_genes) < 100:
        raise ValueError(f"Too few overlapping genes ({len(available_genes)}) for CellTypist!")

    adata_ct = adata_merged[:, available_genes].copy()
    if "symbol_base" in adata_ct.var.columns:
        adata_ct.var_names = pd.Index(adata_ct.var["symbol_base"].values)
        adata_ct.var_names_make_unique()
    sc.pp.normalize_total(adata_ct, target_sum=1e4)
    sc.pp.log1p(adata_ct)

    predictions = celltypist.annotate(adata_ct, model=model, majority_voting=majority_vote, mode="best match")
    pl  = predictions.predicted_labels
    if isinstance(pl, pd.DataFrame):
        pred = pl.get("predicted_labels", pl.iloc[:,0]).astype(str).values
        conf = predictions.probability_matrix.max(axis=1).values
        maj  = pl["majority_voting"].astype(str).values if majority_vote and "majority_voting" in pl.columns else None
    else:
        pred = pl.astype(str).values
        conf = predictions.probability_matrix.max(axis=1).values
        maj  = None

    adata_merged.obs["celltypist_pred"]       = pred
    adata_merged.obs["celltypist_confidence"] = conf
    if maj is not None:
        adata_merged.obs["celltypist_majority"] = maj

    print("  -> CellTypist complete (top 10):")
    print(pd.Series(pred).value_counts().head(10))
    del adata_ct; gc.collect()
    return predictions


def export_celltypist_direct_results(adata_merged, predictions):
    print("  -> Exporting CellTypist direct branch...")
    src_key = "celltypist_majority" if "celltypist_majority" in adata_merged.obs.columns else "celltypist_pred"
    raw = pd.Series(adata_merged.obs[src_key].astype(str).values, index=adata_merged.obs_names)
    adata_merged.obs[CELLTYPIST_DIRECT_LABEL_KEY] = raw.astype("category")

    conf = adata_merged.obs["celltypist_confidence"].values
    filt = np.where(conf >= CELLTYPIST_CONF_THRESHOLD, raw.values, UNLABELED_CATEGORY)
    adata_merged.obs[CELLTYPIST_DIRECT_FILT_KEY] = pd.Categorical(filt)
    n_low = int((conf < CELLTYPIST_CONF_THRESHOLD).sum())
    print(f"    {CELLTYPIST_DIRECT_LABEL_KEY}: {adata_merged.obs[CELLTYPIST_DIRECT_LABEL_KEY].nunique()} types")
    print(f"    {CELLTYPIST_DIRECT_FILT_KEY}:  {n_low} low-conf cells -> '{UNLABELED_CATEGORY}'")

    if CELLTYPIST_SAVE_PROBA:
        proba_mat = predictions.probability_matrix
        proba_df  = proba_mat.copy() if isinstance(proba_mat, pd.DataFrame) else                     pd.DataFrame(np.asarray(proba_mat, dtype=np.float32), index=adata_merged.obs_names)
        if len(proba_df) != adata_merged.n_obs:
            raise ValueError(f"CellTypist proba row mismatch: {len(proba_df)} vs {adata_merged.n_obs}")
        if not proba_df.index.equals(adata_merged.obs_names):
            proba_df = proba_df.reindex(adata_merged.obs_names)
        if proba_df.isna().any().any():
            raise ValueError("NaN in CellTypist probability matrix after reindex!")
        adata_merged.obsm["celltypist_proba"]      = proba_df.values.astype(np.float32)
        adata_merged.uns["celltypist_label_order"] = [str(x) for x in proba_df.columns]
        print(f"    celltypist_proba saved: {proba_df.shape}")

    adata_merged.uns["celltypist_direct"] = {
        "source_column":  src_key,
        "label_key":      CELLTYPIST_DIRECT_LABEL_KEY,
        "filtered_key":   CELLTYPIST_DIRECT_FILT_KEY,
        "conf_threshold": CELLTYPIST_CONF_THRESHOLD,
        "n_low_conf":     n_low,
    }


def run_query_only_leiden(adata_merged, resolution=QUERY_LEIDEN_RESOLUTION):
    print(f"\n[Leiden] Query-only clustering (resolution={resolution})...")
    qmask  = adata_merged.obs["data_source"] == "query"
    qcells = adata_merged.obs_names[qmask]
    if len(qcells) < 10:
        adata_merged.obs["leiden_query"] = "N/A"; return
    X_q = adata_merged.obsm["X_scVI"][qmask.values]
    ad_ = sc.AnnData(X=X_q, obs=adata_merged.obs.loc[qcells].copy())
    ad_.obsm["X_scVI"] = X_q
    sc.pp.neighbors(ad_, use_rep="X_scVI", n_neighbors=30, random_state=RANDOM_SEED)
    sc.tl.leiden(ad_, resolution=resolution, random_state=RANDOM_SEED)
    lf = pd.Series("N/A", index=adata_merged.obs_names, dtype="object")
    lf.loc[qcells] = "qry_" + ad_.obs["leiden"].astype(str)
    adata_merged.obs["leiden_query"] = lf.values
    print(f"  -> {ad_.obs['leiden'].nunique()} query-only clusters")
    del ad_; gc.collect()


## Cell 3 — Initialization

In [4]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")


Output directory: /home/h2048/data/py/20260310/tcell_only_merged_pipeline_v2


## Cell 4 — Step 1: Load

In [5]:
print("\n" + "=" * 80)
print("[Step 1] Loading...")
print("=" * 80)
t0 = time.time()
adata_ref = sc.read_h5ad(REFERENCE_H5AD); adata_ref.var_names_make_unique()
adata_qry = sc.read_h5ad(QUERY_H5AD);     adata_qry.var_names_make_unique()
print(f"[OK] Loaded in {time.time()-t0:.1f}s")
print(f"  Reference: {adata_ref.shape}")
print(f"  Query:     {adata_qry.shape}")



[Step 1] Loading...
[OK] Loaded in 128.8s
  Reference: (44833, 34252)
  Query:     (145335, 83690)


## Cell 5 — Step 2: Subset Reference (optional)

In [6]:
if INCLUDE_COARSE_TYPES is not None and "cell_type_L2" in adata_ref.obs.columns:
    mask = adata_ref.obs["cell_type_L2"].isin(INCLUDE_COARSE_TYPES)
    adata_ref = adata_ref[mask].copy()
    print(f"  Reference after subset: {adata_ref.shape}")
else:
    print("[Step 2] No subsetting.")


[Step 2] No subsetting.


## Cell 6 — Step 3: Preserve Reference Labels (QC only)

In [7]:
print("\n" + "=" * 80)
print("[Step 3] Preserving reference labels as QC column (NOT used for scANVI training)...")
print("=" * 80)
# scANVI will train on CellTypist labels for ALL cells equally.
if REF_LABEL_FINE in adata_ref.obs.columns:
    adata_ref.obs["cell_type_fine_ref"] = adata_ref.obs[REF_LABEL_FINE].astype(str)
    print(f"  -> cell_type_fine_ref from: {REF_LABEL_FINE}")
    print(adata_ref.obs["cell_type_fine_ref"].value_counts())
else:
    adata_ref.obs["cell_type_fine_ref"] = "unknown"
    print(f"  [WARN] {REF_LABEL_FINE} not found")
adata_qry.obs["cell_type_fine_ref"] = "N/A"



[Step 3] Preserving reference labels as QC column (NOT used for scANVI training)...
  -> cell_type_fine_ref from: cell_type_L3
cell_type_fine_ref
cd16plus_nk_cells_c0                      5023
cd16plus_nk_cells_c1                      4215
trm_cytotoxic_t_cells_c0                  4058
trm_cytotoxic_t_cells_c1                  2993
cd8plus_trm_cytotoxic_t_cells_c0          2509
trm_cytotoxic_t_cells_c2                  2449
tem_effector_helper_t_cells_c0            2005
tem_trm_cytotoxic_t_cells_c0              1932
tem_effector_helper_t_cells_c1            1840
cd16plus_nk_cells_c2                      1725
tem_trm_cytotoxic_t_cells_c1              1720
nk_cells_c0                               1590
cd8plus_trm_cytotoxic_t_cells_c1          1569
cd8plus_tem_trm_cytotoxic_t_cells_c0      1337
cd8plus_trm_cytotoxic_t_cells_c2          1334
nk_cells_c1                               1257
tem_temra_cytotoxic_t_cells_c0             984
cd8plus_tem_temra_cytotoxic_t_cells_c0     823
cd8plus

## Cell 7 — Step 4: Common Genes

In [8]:
print("\n" + "=" * 80)
print("[Step 4] Finding common genes...")
print("=" * 80)
qry_set      = set(adata_qry.var_names)
common_genes = [g for g in adata_ref.var_names if g in qry_set]
print(f"  Ref: {adata_ref.n_vars:,}  Query: {adata_qry.n_vars:,}  Common: {len(common_genes):,}")
if len(common_genes) < 1000:
    raise ValueError(f"Too few common genes ({len(common_genes)})")
adata_ref = adata_ref[:, common_genes].copy()
adata_qry = adata_qry[:, common_genes].copy()



[Step 4] Finding common genes...
  Ref: 34,252  Query: 83,690  Common: 33,749


## Cell 8 — Step 5: Validate Counts

In [9]:
print("\n" + "=" * 80)
print("[Step 5] Validating counts...")
print("=" * 80)
ensure_counts_layer(adata_ref, "counts")
ensure_counts_layer(adata_qry, "counts")



[Step 5] Validating counts...
  [WARN] layers['counts'] not found, checking .X...


  -> Auto-copying .X to layers['counts']


'counts'

## Cell 9 — Step 6: Concatenate

In [10]:
print("\n" + "=" * 80)
print("[Step 6] Concatenating (ref + query symmetrically)...")
print("=" * 80)
for ad, prefix in [(adata_ref, "ref_"), (adata_qry, "qry_")]:
    if BATCH_KEY not in ad.obs.columns:
        print(f"  [WARN] {BATCH_KEY} missing, creating placeholder")
        ad.obs[BATCH_KEY] = "unknown_batch"
    ad.obs[BATCH_KEY] = prefix + ad.obs[BATCH_KEY].astype("string").fillna("unknown_batch")

adata_ref.obs_names = pd.Index([f"ref_{x}" for x in adata_ref.obs_names])
adata_qry.obs_names = pd.Index([f"qry_{x}" for x in adata_qry.obs_names])

adata_merged = sc.concat(
    {"reference": adata_ref, "query": adata_qry},
    axis=0, join="inner", merge="unique", label="data_source"
)
print(f"  [INFO] Merged: {adata_merged.shape}")
print(f"  Reference: {(adata_merged.obs['data_source']=='reference').sum():,}")
print(f"  Query:     {(adata_merged.obs['data_source']=='query').sum():,}")
del adata_ref, adata_qry
gc.collect()



[Step 6] Concatenating (ref + query symmetrically)...


  [INFO] Merged: (190168, 33749)
  Reference: 44,833
  Query:     145,335


46754

## Cell 10 — Step 7: Covariates & symbol_base

In [11]:
print("\n" + "=" * 80)
print("[Step 7] Preparing covariates...")
print("=" * 80)
t0 = time.time()
prepare_covariates(adata_merged)
print(f"[OK] Covariates done in {time.time()-t0:.1f}s")
print("  -> Adding symbol_base column...")
adata_merged.var["symbol_base"] = adata_merged.var_names.str.replace(r"-\d+$", "", regex=True)



[Step 7] Preparing covariates...
  -> Validating counts...
  -> Batch/Tissue...
  -> Signature scores...


  -> Computing T/NK QC scores...
    CD4_single: 33912
    CD8_single: 96722
    DP: 13977
    DN: 45557
[OK] Covariates done in 24.5s
  -> Adding symbol_base column...


## Cell 11 — Step 8: CellTypist on ALL Cells
> CellTypist annotates ref AND query cells with equal weight.
> These labels become scANVI training labels for everyone.

In [12]:
print("\n" + "=" * 80)
print("[Step 8] Running CellTypist on all merged cells...")
print("=" * 80)
import time as _time
_t8 = _time.time()

celltypist_predictions = run_celltypist_on_full_genes(adata_merged)
export_celltypist_direct_results(adata_merged, celltypist_predictions)

print(f"[OK] CellTypist done in {_time.time()-_t8:.1f}s")
print()

# ── Label distribution (all cells) ───────────────────────────────────────────
_ct_counts = adata_merged.obs[CELLTYPIST_DIRECT_LABEL_KEY].value_counts()
n_unknown   = (adata_merged.obs[CELLTYPIST_DIRECT_FILT_KEY] == UNLABELED_CATEGORY).sum()

print(f"[INFO] Total cells: {adata_merged.n_obs:,}")
print(f"[INFO] Unique labels (direct):    {_ct_counts.shape[0]}")
print(f"[INFO] Unknown after conf filter: {n_unknown:,} ({n_unknown/adata_merged.n_obs:.1%})")
print()
print("[INFO] Label distribution (all cells):")
for _lbl, _n in _ct_counts.items():
    print(f"  {_lbl:<40} {_n:>7,}  ({_n/adata_merged.n_obs*100:.1f}%)")

print()
print("[INFO] Label distribution by data_source:")
for _src in adata_merged.obs["data_source"].unique():
    _mask = adata_merged.obs["data_source"] == _src
    _sub  = adata_merged.obs.loc[_mask, CELLTYPIST_DIRECT_LABEL_KEY].value_counts()
    print(f"  [{_src}]  n={_mask.sum():,}")
    for _lbl, _n in _sub.items():
        print(f"    {_lbl:<40} {_n:>7,}  ({_n/_mask.sum()*100:.1f}%)")
    print()



[Step 8] Running CellTypist on all merged cells...

[CellTypist] Annotating ALL cells (ref + query)...
  -> Loaded: /home/h2048/data/source/reference/celltypist_models/Immune_All_Low.pkl
  -> 6088/6639 genes matched via symbol_base


🔬 Input data has 190168 cells and 6088 genes
🔗 Matching reference genes in the model
🧬 6088 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 25
🗳️ Majority voting the predictions
✅ Majority voting done!


  -> CellTypist complete (top 10):
Tem/Trm cytotoxic T cells      34447
Trm cytotoxic T cells          25678
Regulatory T cells             24160
CD16+ NK cells                 17570
Tem/Effector helper T cells    15675
Tem/Temra cytotoxic T cells    13759
CD16- NK cells                  8353
Tcm/Naive helper T cells        7819
Type 1 helper T cells           6764
NK cells                        6490
Name: count, dtype: int64
  -> Exporting CellTypist direct branch...
    celltypist_label_direct: 20 types
    celltypist_label_direct_filt:  60931 low-conf cells -> 'Unknown'
    celltypist_proba saved: (190168, 98)
[OK] CellTypist done in 334.2s

[INFO] Total cells: 190,168
[INFO] Unique labels (direct):    20
[INFO] Unknown after conf filter: 60,931 (32.0%)

[INFO] Label distribution (all cells):
  Trm cytotoxic T cells                     45,020  (23.7%)
  Tem/Trm cytotoxic T cells                 36,695  (19.3%)
  Regulatory T cells                        25,471  (13.4%)
  Tem/Effect

## Cell 8.5 — Step 8.5: CD4/CD8 Scoring & Visualization

In [13]:
print("\n" + "=" * 80)
print("[Step 8.5] CD4/CD8 scoring and pre-scANVI visualization")
print("=" * 80)
import time as _time
_t85 = _time.time()

# =============================================================================
# Step 7 (prepare_covariates -> compute_tnk_scores) already computed
# CD4_score_norm / CD8_score_norm / NK_score_norm from raw counts via
# normalize_total(1e4) + log1p on a per-gene-panel temporary AnnData.
# We reuse those columns here — no additional normalization needed.
# =============================================================================

_required = ["CD4_score_norm", "CD8_score_norm", "NK_score_norm",
             "CD4_score",      "CD8_score",      "NK_score"]
_missing = [c for c in _required if c not in adata_merged.obs.columns]
if _missing:
    raise RuntimeError(
        f"[Step 8.5] Missing score columns: {_missing}\n"
        "  -> Step 7 (prepare_covariates / compute_tnk_scores) must be run first."
    )

print("[INFO] Reusing normalized scores from Step 7 (compute_tnk_scores):")
for col in _required:
    v = adata_merged.obs[col]
    print(f"  {col:<22} mean={v.mean():.4f}  min={v.min():.4f}  max={v.max():.4f}")
print()

# ── Score-based classification (using _norm columns for threshold comparison) ─
# _norm columns are min-max scaled [0, 1] within each score,
# so thresholds here are on the 0-1 scale.
CD4_SCORE_THRESH = 0.3   # edit if violin plots suggest adjustment
CD8_SCORE_THRESH = 0.3

def _classify(row):
    is_cd4 = row["CD4_score_norm"] > CD4_SCORE_THRESH
    is_cd8 = row["CD8_score_norm"] > CD8_SCORE_THRESH
    if   is_cd4 and is_cd8: return "DP"
    elif is_cd4:             return "CD4_single"
    elif is_cd8:             return "CD8_single"
    else:                    return "DN"

adata_merged.obs["cd4_cd8_by_score"] = (
    adata_merged.obs[["CD4_score_norm", "CD8_score_norm"]].apply(_classify, axis=1)
    .astype("category")
)
print("[INFO] Score classification (thresh CD4={}, CD8={}):".format(
    CD4_SCORE_THRESH, CD8_SCORE_THRESH))
print(adata_merged.obs["cd4_cd8_by_score"].value_counts().to_string())
print()

# ── Exact CellTypist label -> lineage mapping ─────────────────────────────────
CT_LABEL_LINEAGE = {
    # CD4 lineage
    "Regulatory T cells":          "CD4",
    "Tcm/Naive helper T cells":    "CD4",
    "Effector helper T cells":     "CD4",
    # CD8 lineage
    "Tem/Temra cytotoxic T cells": "CD8",
    "Tem/Trm cytotoxic T cells":   "CD8",
    "Trm cytotoxic T cells":       "CD8",
    # NK
    "NK cells":                    "NK",
    "CD16+ NK cells":              "NK",
    "CD16- NK cells":              "NK",
    # Ambiguous / non-target
    "CRTAM+ gamma-delta T cells":  "gdT",
    "Cycling T cells":             "cycling",
    "MAIT cells":                  "MAIT",
    "ILC3":                        "ILC",
    "Mast cells":                  "non_immune",
    "Endothelial cells":           "non_immune",
    "Epithelial cells":            "non_immune",
    "Memory B cells":              "non_immune",
    "Naive B cells":               "non_immune",
    "Plasma cells":                "non_immune",
}

_ct_raw = adata_merged.obs[CELLTYPIST_DIRECT_FILT_KEY].astype(str)
adata_merged.obs["_ct_lineage"] = _ct_raw.map(
    lambda x: CT_LABEL_LINEAGE.get(x, "other")
)

unseen = set(_ct_raw.unique()) - set(CT_LABEL_LINEAGE.keys()) - {"Unknown"}
if unseen:
    print(f"  [WARN] {len(unseen)} labels not in CT_LABEL_LINEAGE (mapped to 'other'):")
    for lb in sorted(unseen):
        print(f"    '{lb}': {(_ct_raw == lb).sum():,} cells")
    print()

print("[INFO] CellTypist lineage summary:")
print(adata_merged.obs["_ct_lineage"].value_counts().to_string())
print()

# ── Conflict detection ────────────────────────────────────────────────────────
adata_merged.obs["cd4_cd8_conflict"] = "none"

mask_cd4_ct    = adata_merged.obs["_ct_lineage"] == "CD4"
mask_cd8_ct    = adata_merged.obs["_ct_lineage"] == "CD8"
mask_score_cd4 = adata_merged.obs["cd4_cd8_by_score"] == "CD4_single"
mask_score_cd8 = adata_merged.obs["cd4_cd8_by_score"] == "CD8_single"
mask_score_dn  = adata_merged.obs["cd4_cd8_by_score"] == "DN"

adata_merged.obs.loc[mask_cd4_ct & mask_score_cd8, "cd4_cd8_conflict"] = "CT=CD4_but_score=CD8"
adata_merged.obs.loc[mask_cd8_ct & mask_score_cd4, "cd4_cd8_conflict"] = "CT=CD8_but_score=CD4"
adata_merged.obs.loc[(mask_cd4_ct | mask_cd8_ct) & mask_score_dn, "cd4_cd8_conflict"] = "score_DN"

print("[INFO] Conflict summary:")
print(adata_merged.obs["cd4_cd8_conflict"].value_counts().to_string())
print()

# ── Visualizations ─────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

output_dir.mkdir(parents=True, exist_ok=True)
fig_dir = output_dir / "figures"
fig_dir.mkdir(exist_ok=True)

# Ordered label list for plots
_lineage_order = ["CD4", "CD8", "NK", "gdT", "MAIT", "ILC", "cycling", "non_immune", "other"]
_label_order = []
for _lin in _lineage_order:
    _label_order += sorted(
        [lb for lb, lin in CT_LABEL_LINEAGE.items()
         if lin == _lin
         and lb in adata_merged.obs[CELLTYPIST_DIRECT_LABEL_KEY].cat.categories]
    )
_all_cats = list(adata_merged.obs[CELLTYPIST_DIRECT_LABEL_KEY].cat.categories)
_label_order += [lb for lb in _all_cats if lb not in _label_order]

# 1. Violin: CD4_score_norm / CD8_score_norm / NK_score_norm per CellTypist label
#    Only on CD4/CD8/NK cells to keep plot readable
_mask_rel = adata_merged.obs["_ct_lineage"].isin(["CD4", "CD8", "NK"])
if _mask_rel.sum() > 0:
    _sub = adata_merged[_mask_rel].copy()
    _sub_order = [lb for lb in _label_order
                  if lb in _sub.obs[CELLTYPIST_DIRECT_LABEL_KEY].cat.categories]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (score_col, title, color) in zip(axes, [
        ("CD4_score_norm", "CD4 Score (norm) by CellTypist Label", "#d62728"),
        ("CD8_score_norm", "CD8 Score (norm) by CellTypist Label", "#1f77b4"),
        ("NK_score_norm",  "NK Score (norm) by CellTypist Label",  "#2ca02c"),
    ]):
        sc.pl.violin(_sub, keys=score_col,
                     groupby=CELLTYPIST_DIRECT_LABEL_KEY,
                     order=_sub_order, ax=ax, show=False,
                     rotation=35, color=color)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("")
        ax.axhline(CD4_SCORE_THRESH if "CD4" in score_col else CD8_SCORE_THRESH,
                   color="black", linestyle="--", lw=0.8, label="threshold")
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(str(fig_dir / "step85_violin_scores_by_label.pdf"),
                dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close()
    print(f"[OK] Violin saved: step85_violin_scores_by_label.pdf")
    del _sub

# 2. Dotplot: key lineage markers per CellTypist label
_use_raw = adata_merged.raw is not None
_vnames  = adata_merged.raw.var_names if _use_raw else adata_merged.var_names
dotplot_genes = ["CD4", "IL7R", "FOXP3", "IL2RA",
                 "CD8A", "CD8B",
                 "GZMB", "GZMK", "PRF1", "GNLY",
                 "NKG7", "NCAM1", "KLRB1", "FCGR3A",
                 "CCR7", "TCF7", "SELL",
                 "PDCD1", "LAG3", "HAVCR2",
                 "TBX21", "IFNG", "MKI67", "TOP2A"]
_dp_genes = [g for g in dotplot_genes if g in _vnames]

if _dp_genes and _label_order:
    try:
        dp = sc.pl.dotplot(
            adata_merged, var_names=_dp_genes,
            groupby=CELLTYPIST_DIRECT_LABEL_KEY,
            categories_order=_label_order,
            use_raw=_use_raw, standard_scale="var",
            cmap="Reds", show=False, return_fig=True,
            figsize=(max(14, len(_dp_genes)*0.6), max(6, len(_label_order)*0.38)),
            dendrogram=False,
        )
        dp.savefig(str(fig_dir / "step85_dotplot_cd4cd8_by_celltypist.pdf"),
                   dpi=FIGURE_DPI, bbox_inches="tight")
        plt.close("all")
        print(f"[OK] Dotplot saved: step85_dotplot_cd4cd8_by_celltypist.pdf")
    except Exception as e:
        print(f"  [WARN] Dotplot failed: {e}")

# 3. UMAP (only if already available — otherwise deferred to Step 16)
if "X_umap" not in adata_merged.obsm:
    print("[INFO] X_umap not yet computed — UMAP panels deferred to Step 16")
else:
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    for ax, (key, title, cmap_) in zip(axes, [
        ("CD4_score_norm",   "CD4 Score (norm)",        "RdBu_r"),
        ("CD8_score_norm",   "CD8 Score (norm)",        "RdBu_r"),
        ("cd4_cd8_by_score", "CD4/CD8 by Score",        None),
        ("cd4_cd8_conflict", "CT vs Score Conflict",    None),
    ]):
        kw = dict(ax=ax, show=False, frameon=False, size=2, alpha=0.5,
                  rasterized=True, title=title)
        if cmap_: kw["color_map"] = cmap_
        sc.pl.umap(adata_merged, color=key, **kw)
    plt.tight_layout()
    plt.savefig(str(fig_dir / "step85_umap_cd4cd8_scores.pdf"),
                dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close()
    print(f"[OK] UMAP panels saved: step85_umap_cd4cd8_scores.pdf")

print(f"[OK] Step 8.5 done in {_time.time()-_t85:.1f}s")



[Step 8.5] CD4/CD8 scoring and pre-scANVI visualization
[INFO] Reusing normalized scores from Step 7 (compute_tnk_scores):
  CD4_score_norm         mean=0.1693  min=0.0000  max=1.0000
  CD8_score_norm         mean=0.3459  min=0.0000  max=1.0000
  NK_score_norm          mean=0.2465  min=0.0000  max=1.0000
  CD4_score              mean=1.2211  min=0.0000  max=7.2119
  CD8_score              mean=2.5013  min=0.0000  max=7.2307
  NK_score               mean=1.8190  min=0.0000  max=7.3806

[INFO] Score classification (thresh CD4=0.3, CD8=0.3):
cd4_cd8_by_score
CD8_single    96722
DN            45557
CD4_single    33912
DP            13977

  [WARN] 5 labels not in CT_LABEL_LINEAGE (mapped to 'other'):
    'Follicular helper T cells': 969 cells
    'Tem/Effector helper T cells': 12,670 cells
    'Type 1 helper T cells': 1,021 cells
    'Type 17 helper T cells': 1,952 cells
    'gamma-delta T cells': 528 cells

[INFO] CellTypist lineage summary:
_ct_lineage
other         78071
CD8           

... storing 'cell_type_fine_ref' as categorical
... storing 'phase' as categorical
... storing 'celltypist_pred' as categorical
... storing 'celltypist_majority' as categorical
... storing '_ct_lineage' as categorical
... storing 'cd4_cd8_conflict' as categorical
... storing 'symbol_base' as categorical


[OK] Violin saved: step85_violin_scores_by_label.pdf


... storing 'cell_type_fine_ref' as categorical
... storing 'phase' as categorical
... storing 'celltypist_pred' as categorical
... storing 'celltypist_majority' as categorical
... storing '_ct_lineage' as categorical
... storing 'cd4_cd8_conflict' as categorical
... storing 'symbol_base' as categorical


[OK] Dotplot saved: step85_dotplot_cd4cd8_by_celltypist.pdf
[INFO] X_umap not yet computed — UMAP panels deferred to Step 16
[OK] Step 8.5 done in 49.1s


## Cell 8.6 — Step 8.6: MANUAL REVIEW — Reconcile scanvi_labels with CD4/CD8 scores

**Review the dotplot from Step 8.5, then choose a strategy and edit this cell.**

| Strategy | Effect |
|---|---|
| `"conflict_to_unknown"` | Cells where CellTypist lineage conflicts with score → `Unknown` |
| `"score_override"` | Replace CellTypist label with score-derived coarse label |
| `"keep_celltypist"` | No change (skip reconciliation) |

After setting the strategy, rerun this cell; it rewrites `scanvi_labels` in-place.  
Proceed to Step 9 when satisfied.

In [14]:
print("\n" + "=" * 80)
print("[Step 8.6] MANUAL REVIEW — CD4/CD8 label reconciliation")
print("=" * 80)

# =============================================================================
# EDIT BELOW
# =============================================================================

# Strategy: "conflict_to_unknown" | "score_override" | "keep_celltypist"
CD4_CD8_OVERRIDE_STRATEGY = "conflict_to_unknown"

# Which conflict types to act on (leave empty list [] to act on all)
CONFLICT_TYPES_TO_FIX = [
    "CT=CD4_but_score=CD8",
    "CT=CD8_but_score=CD4",
]

# Score thresholds for "score_override" — override only when score is high-confidence
CD4_OVERRIDE_THRESH = 0.5
CD8_OVERRIDE_THRESH = 0.5

# =============================================================================
# APPLY (do not edit below this line)
# =============================================================================


# Start from CellTypist-filtered labels (same as Step 11 would do)
if "scanvi_labels_reconciled" not in adata_merged.obs.columns:
    adata_merged.obs["scanvi_labels_reconciled"] = (
        adata_merged.obs[CELLTYPIST_DIRECT_FILT_KEY].copy()
    )

n_changed = 0

if CD4_CD8_OVERRIDE_STRATEGY == "keep_celltypist":
    print("[INFO] Strategy = keep_celltypist — no changes applied.")

elif CD4_CD8_OVERRIDE_STRATEGY == "conflict_to_unknown":
    conflict_col = adata_merged.obs["cd4_cd8_conflict"]
    target_conflicts = (
        CONFLICT_TYPES_TO_FIX if CONFLICT_TYPES_TO_FIX else conflict_col.unique().tolist()
    )
    for ct in target_conflicts:
        mask = conflict_col == ct
        n = mask.sum()
        if n > 0:
            adata_merged.obs.loc[mask, "scanvi_labels_reconciled"] = UNLABELED_CATEGORY
            print(f"  -> Set {n:,} cells ({ct}) to Unknown")
            n_changed += n
    print(f"[OK] conflict_to_unknown: {n_changed:,} cells set to Unknown")

elif CD4_CD8_OVERRIDE_STRATEGY == "score_override":
    # Overwrite label only when score clearly high-confidence
    mask_hi_cd4 = (
        (adata_merged.obs["CD4_score_norm"] >= CD4_OVERRIDE_THRESH) &
        (adata_merged.obs["_ct_lineage"] == "CD8")
    )
    mask_hi_cd8 = (
        (adata_merged.obs["CD8_score_norm"] >= CD8_OVERRIDE_THRESH) &
        (adata_merged.obs["_ct_lineage"] == "CD4")
    )
    adata_merged.obs.loc[mask_hi_cd4, "scanvi_labels_reconciled"] = "CD4+ T"
    adata_merged.obs.loc[mask_hi_cd8, "scanvi_labels_reconciled"] = "CD8+ T"
    n_changed = mask_hi_cd4.sum() + mask_hi_cd8.sum()
    print(f"[OK] score_override: {n_changed:,} cells relabeled "
          f"(CD4→CD8: {mask_hi_cd8.sum():,}  CD8→CD4: {mask_hi_cd4.sum():,})")

else:
    raise ValueError(f"Unknown strategy: {CD4_CD8_OVERRIDE_STRATEGY!r}. "
                     "Choose conflict_to_unknown | score_override | keep_celltypist")

# Summary
print()
print("[INFO] scanvi_labels_reconciled distribution:")
print(adata_merged.obs["scanvi_labels_reconciled"].value_counts().to_string())
print()
print("[INFO] Unknown fraction: "
      f"{(adata_merged.obs['scanvi_labels_reconciled'] == UNLABELED_CATEGORY).mean():.2%}")

# ── Make this the active training label for Step 11 ──────────────────────────
# Step 11 will look for SCANVI_LABEL_RECONCILED_KEY = "scanvi_labels_reconciled"
# The existing SCANVI_LABEL_SOURCE logic in Step 11 is overridden when this key exists.
SCANVI_LABEL_RECONCILED_KEY = "scanvi_labels_reconciled"
print(f"[OK] Active scANVI label column set to: \"{SCANVI_LABEL_RECONCILED_KEY}\"")
print("     -> Proceed to Step 9 when satisfied with the distribution above.")



[Step 8.6] MANUAL REVIEW — CD4/CD8 label reconciliation
  -> Set 890 cells (CT=CD4_but_score=CD8) to Unknown
  -> Set 1,758 cells (CT=CD8_but_score=CD4) to Unknown
[OK] conflict_to_unknown: 2,648 cells set to Unknown

[INFO] scanvi_labels_reconciled distribution:
scanvi_labels_reconciled
Unknown                        63579
Tem/Trm cytotoxic T cells      25190
Trm cytotoxic T cells          23215
Regulatory T cells             20089
CD16+ NK cells                 17070
Tem/Effector helper T cells    12670
Tem/Temra cytotoxic T cells     8236
CD16- NK cells                  6225
Tcm/Naive helper T cells        3528
NK cells                        2237
Type 17 helper T cells          1952
CRTAM+ gamma-delta T cells      1187
Type 1 helper T cells           1021
Follicular helper T cells        969
MAIT cells                       899
ILC3                             795
gamma-delta T cells              528
Cycling T cells                  257
Endothelial cells                238
Epithel

## Cell 12 — Step 9: HVG Selection

In [15]:
print("\n" + "=" * 80)
print("[Step 9] Selecting HVGs...")
print("=" * 80)
hvg_method = "unknown"
try:
    sc.pp.highly_variable_genes(adata_merged, layer="counts", n_top_genes=N_HVG,
                                batch_key=BATCH_KEY, flavor="seurat_v3", subset=False)
    hvg_method = "batch_seurat_v3"
except Exception as e1:
    print(f"  -> batch-aware failed ({str(e1)[:50]}), trying standard...")
    try:
        sc.pp.highly_variable_genes(adata_merged, layer="counts", n_top_genes=N_HVG,
                                    flavor="seurat_v3", subset=False)
        hvg_method = "standard_seurat_v3"
    except Exception as e2:
        print(f"  -> seurat_v3 failed ({str(e2)[:50]}), fallback to cell_ranger")
        sc.pp.highly_variable_genes(adata_merged, layer="counts", n_top_genes=N_HVG,
                                    flavor="cell_ranger", subset=False)
        hvg_method = "cell_ranger"
print(f"  -> Method: {hvg_method}")

if FORCE_MARKERS_IN_HVG:
    mset, n_added = set(FORCED_MARKERS), 0
    for idx, sb in enumerate(adata_merged.var["symbol_base"]):
        if sb in mset:
            rn = adata_merged.var_names[idx]
            if not adata_merged.var.loc[rn, "highly_variable"]:
                adata_merged.var.loc[rn, "highly_variable"] = True
                n_added += 1
    print(f"  -> Forced {n_added} markers into HVG")

n_hvg_final = int(adata_merged.var["highly_variable"].sum())
print(f"  -> Final HVG count: {n_hvg_final}")
hvg_genes = adata_merged.var_names[adata_merged.var["highly_variable"]].tolist()
(output_dir / f"{OUTPUT_PREFIX}_hvg_genes.txt").write_text("\n".join(hvg_genes))



[Step 9] Selecting HVGs...


  -> batch-aware failed (b'There are other near singularities as well. 0.09), trying standard...
  -> Method: standard_seurat_v3
  -> Forced 11 markers into HVG
  -> Final HVG count: 4011


31284

## Cell 13 — Step 10: Preserve Full Matrix for .raw

In [16]:
print("\n" + "=" * 80)
print("[Step 10] Preserving full matrix for .raw...")
print("=" * 80)
full_counts = adata_merged.layers["counts"]
if issparse(full_counts) and not isinstance(full_counts, csr_matrix):
    full_counts = csr_matrix(full_counts)
raw_var = adata_merged.var.copy()
print(f"  Full matrix: {full_counts.shape}")

# Checkpoint
adata_merged.write_h5ad(
    str(output_dir / f"{OUTPUT_PREFIX}_adata_preprocessed.h5ad"),
    compression="gzip"
)
print(f"[OK] Checkpoint saved: {OUTPUT_PREFIX}_adata_preprocessed.h5ad")



[Step 10] Preserving full matrix for .raw...
  Full matrix: (190168, 33749)
[OK] Checkpoint saved: tcell_merged_v2_adata_preprocessed.h5ad


## Cell 14 — Step 11: Build Training Subset
> **Architecture v2.0:** `scanvi_labels` = CellTypist labels for ALL cells.
> Query is NOT forced to Unknown. Only genuinely low-confidence cells become Unknown.

In [17]:
print("\n" + "=" * 80)
print("[Step 11] Building HVG training subset...")
print("=" * 80)
hvg_mask = adata_merged.var["highly_variable"].values
X_hvg    = adata_merged.layers["counts"][:, hvg_mask]
if issparse(X_hvg) and not isinstance(X_hvg, csr_matrix):
    X_hvg = csr_matrix(X_hvg)

adata_train = sc.AnnData(
    X=X_hvg.copy(),
    obs=adata_merged.obs.copy(),
    var=adata_merged.var.iloc[hvg_mask].copy()
)
adata_train.var_names        = adata_merged.var_names[hvg_mask]
adata_train.layers["counts"] = adata_train.X
print(f"  [INFO] Training data: {adata_train.shape}")

# =============================================================================
# ARCHITECTURE v2.1: if Step 8.6 was run, prefer reconciled labels.
if "scanvi_labels_reconciled" in adata_merged.obs.columns:
    label_source_col = "scanvi_labels_reconciled"
    print(f"[INFO] Using reconciled labels from Step 8.6: {label_source_col}")
else:
    # fall through to original SCANVI_LABEL_SOURCE logic below
    pass
# ARCHITECTURE v2.0: scanvi_labels from CellTypist for ALL cells equally.
# SCANVI_LABEL_SOURCE = "filtered" -> celltypist_label_direct_filt
#                                     (low-conf cells already Unknown)
# SCANVI_LABEL_SOURCE = "direct"   -> celltypist_label_direct (unfiltered)
# =============================================================================
label_source_col = (
    CELLTYPIST_DIRECT_FILT_KEY if SCANVI_LABEL_SOURCE == "filtered"
    else CELLTYPIST_DIRECT_LABEL_KEY
)
print(f"  -> scANVI label source: {label_source_col}")
print(f"     All cells (ref + query) labeled by CellTypist — no asymmetry")

adata_train.obs["scanvi_labels"] = adata_train.obs[label_source_col].astype(str).astype("category")
if UNLABELED_CATEGORY not in adata_train.obs["scanvi_labels"].cat.categories:
    adata_train.obs["scanvi_labels"] = adata_train.obs["scanvi_labels"].cat.add_categories([UNLABELED_CATEGORY])

print("\n  -> scANVI training label distribution:")
print(adata_train.obs["scanvi_labels"].value_counts())
n_unknown = (adata_train.obs["scanvi_labels"] == UNLABELED_CATEGORY).sum()
print(f"\n  -> Unknown (low-conf): {n_unknown:,} / {adata_train.n_obs:,} "
      f"({100*n_unknown/adata_train.n_obs:.1f}%)")
gc.collect()



[Step 11] Building HVG training subset...


  [INFO] Training data: (190168, 4011)
[INFO] Using reconciled labels from Step 8.6: scanvi_labels_reconciled
  -> scANVI label source: celltypist_label_direct_filt
     All cells (ref + query) labeled by CellTypist — no asymmetry

  -> scANVI training label distribution:
scanvi_labels
Unknown                        60931
Tem/Trm cytotoxic T cells      26384
Trm cytotoxic T cells          23694
Regulatory T cells             20592
CD16+ NK cells                 17070
Tem/Effector helper T cells    12670
Tem/Temra cytotoxic T cells     8321
CD16- NK cells                  6225
Tcm/Naive helper T cells        3915
NK cells                        2237
Type 17 helper T cells          1952
CRTAM+ gamma-delta T cells      1187
Type 1 helper T cells           1021
Follicular helper T cells        969
MAIT cells                       899
ILC3                             795
gamma-delta T cells              528
Cycling T cells                  257
Endothelial cells                238
Epithelial

823

## Cell 15 — Step 12: scVI Training

In [18]:
print("\n" + "=" * 80)
print("[Step 12] Training scVI...")
print("=" * 80)
scvi.model.SCVI.setup_anndata(
    adata_train,
    layer="counts",
    batch_key=BATCH_KEY,
    continuous_covariate_keys=["pct_counts_mt","stress_score","S_score","G2M_score"],
    categorical_covariate_keys=[TISSUE_KEY]
)
scvi_model = scvi.model.SCVI(
    adata_train,
    n_latent=SCVI_N_LATENT, n_layers=SCVI_N_LAYERS,
    n_hidden=SCVI_N_HIDDEN, dropout_rate=SCVI_DROPOUT
)
train_kwargs = {
    "max_epochs": MAX_EPOCHS_SCVI, "batch_size": BATCH_SIZE,
    "early_stopping": True, "early_stopping_patience": 30,
    "plan_kwargs": {"lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
}
if gpu_available:
    train_kwargs["accelerator"] = "gpu"; train_kwargs["devices"] = 1
t0 = time.time()
scvi_model.train(**train_kwargs)
print(f"[OK] scVI training complete  ({time.time()-t0:.1f}s)")


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



[Step 12] Training scVI...
Epoch 1/400:   0%|          | 0/400 [00:00<?, ?it/s]

Epoch 389/400:  97%|█████████▋| 389/400 [2:31:48<04:17, 23.41s/it, v_num=1, train_loss_step=734, train_loss_epoch=658]  
Monitored metric elbo_validation did not improve in the last 30 records. Best score: 663.147. Signaling Trainer to stop.
[OK] scVI training complete  (9108.5s)


## Cell 16 — Step 13: scANVI Training

In [19]:
print("\n" + "=" * 80)
print("[Step 13] Training scANVI (CellTypist labels for all cells)...")
print("=" * 80)
scanvi_model = scvi.model.SCANVI.from_scvi_model(
    scvi_model, adata=adata_train,
    labels_key="scanvi_labels",
    unlabeled_category=UNLABELED_CATEGORY
)
scanvi_train_kwargs = {
    "max_epochs": MAX_EPOCHS_SCANVI, "batch_size": BATCH_SIZE,
    "early_stopping": True, "early_stopping_patience": 20,
    "plan_kwargs": {"lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
}
if gpu_available:
    scanvi_train_kwargs["accelerator"] = "gpu"; scanvi_train_kwargs["devices"] = 1
t0 = time.time()
scanvi_model.train(**scanvi_train_kwargs)
print(f"[OK] scANVI training complete  ({time.time()-t0:.1f}s)")



[Step 13] Training scANVI (CellTypist labels for all cells)...
INFO     Training for 200 epochs.                                                                                  


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 193/200:  96%|█████████▋| 193/200 [2:42:47<05:54, 50.61s/it, v_num=1, train_loss_step=660, train_loss_epoch=653]  
Monitored metric elbo_validation did not improve in the last 20 records. Best score: 667.835. Signaling Trainer to stop.
[OK] scANVI training complete  (9768.2s)


## Cell 17 — Step 14: Export Latent & Predictions

In [20]:
print("\n" + "=" * 80)
print("[Step 14] Exporting latent representations and predictions...")
print("=" * 80)

lat    = scanvi_model.get_latent_representation(adata_train)
lat_df = pd.DataFrame(lat, index=adata_train.obs_names,
                      columns=[f"scANVI_{i}" for i in range(lat.shape[1])])
lat_al = lat_df.reindex(adata_merged.obs_names)
if lat_al.isna().any().any():
    raise ValueError("Missing scANVI latent after reindex!")
adata_merged.obsm["X_scANVI"] = lat_al.values

pred_al = pd.Series(scanvi_model.predict(adata_train), index=adata_train.obs_names
                    ).reindex(adata_merged.obs_names)
adata_merged.obs["scanvi_pred"] = pred_al.values

proba_raw = scanvi_model.predict(adata_train, soft=True)
if isinstance(proba_raw, pd.DataFrame):
    label_order = list(proba_raw.columns)
    proba       = proba_raw.values.astype(np.float32)
else:
    proba = np.asarray(proba_raw, dtype=np.float32)
    try:
        label_order = list(scanvi_model.adata_manager.get_state_registry("labels").categorical_mapping)
    except Exception:
        label_order = [f"label_{i}" for i in range(proba.shape[1])]

proba_df = pd.DataFrame(proba, index=adata_train.obs_names, columns=label_order)
proba_al = proba_df.reindex(adata_merged.obs_names)
adata_merged.obsm["scanvi_proba"]     = proba_al.values
adata_merged.obs["scanvi_confidence"] = proba_al.values.max(axis=1)
adata_merged.uns["scanvi_label_order"]= list(label_order)

print("  -> scANVI predictions (top 15):")
print(adata_merged.obs["scanvi_pred"].value_counts().head(15))

if not np.isfinite(proba_al.values).all():
    raise ValueError("Non-finite values in scANVI probability matrix!")
ents = entropy(proba_al.values + 1e-10, axis=1)
adata_merged.obs["scanvi_entropy"] = ents
emin, emax = ents.min(), ents.max()
adata_merged.obs["novelty_score"] = (ents - emin)/(emax - emin) if emax > emin else 0.0
qmask = adata_merged.obs["data_source"] == "query"
adata_merged.obs["is_potentially_novel"] = (adata_merged.obs["novelty_score"] > 0.7) & qmask
print(f"  -> High novelty query cells: {adata_merged.obs['is_potentially_novel'].sum()}")



[Step 14] Exporting latent representations and predictions...


  -> scANVI predictions (top 15):
scanvi_pred
Regulatory T cells             74221
Trm cytotoxic T cells          35372
Tem/Trm cytotoxic T cells      29623
CD16+ NK cells                 20836
Tem/Temra cytotoxic T cells    16004
CD16- NK cells                 14112
Name: count, dtype: int64
  -> High novelty query cells: 83221


## Cell 18 — Step 15: Attach .raw

In [21]:
print("\n" + "=" * 80)
print("[Step 15] Attaching .raw...")
print("=" * 80)
from anndata import AnnData
adata_merged.raw = AnnData(X=full_counts, obs=adata_merged.obs.copy(), var=raw_var)
print(f"  [OK] .raw: {adata_merged.raw.n_vars} genes")



[Step 15] Attaching .raw...
  [OK] .raw: 33749 genes


## Cell 19 — Step 16: Multiple UMAPs

In [22]:
print("\n" + "=" * 80)
print("[Step 16] Computing UMAPs...")
print("=" * 80)

lat_s    = scvi_model.get_latent_representation(adata_train)
lat_s_df = pd.DataFrame(lat_s, index=adata_train.obs_names,
                         columns=[f"scVI_{i}" for i in range(lat_s.shape[1])])
lat_s_al = lat_s_df.reindex(adata_merged.obs_names)
if lat_s_al.isna().any().any():
    raise ValueError("Missing scVI latent after reindex!")
adata_merged.obsm["X_scVI"] = lat_s_al.values

run_query_only_leiden(adata_merged, resolution=QUERY_LEIDEN_RESOLUTION)

sc.pp.neighbors(adata_merged, use_rep="X_scVI",   n_neighbors=30,
                random_state=RANDOM_SEED, key_added="neighbors_scVI")
sc.tl.umap(adata_merged, random_state=RANDOM_SEED, neighbors_key="neighbors_scVI")
adata_merged.obsm["X_umap_scVI"] = adata_merged.obsm["X_umap"].copy()
print("  -> X_umap_scVI saved")

sc.pp.neighbors(adata_merged, use_rep="X_scANVI", n_neighbors=30,
                random_state=RANDOM_SEED, key_added="neighbors_scANVI")
sc.tl.umap(adata_merged, random_state=RANDOM_SEED, neighbors_key="neighbors_scANVI")
adata_merged.obsm["X_umap_scANVI"] = adata_merged.obsm["X_umap"].copy()
adata_merged.obsm["X_umap"]        = adata_merged.obsm["X_umap_scANVI"].copy()
print("  -> X_umap_scANVI saved (default X_umap)")

umap_op = UMAP(n_neighbors=30, n_components=2, min_dist=0.5,
               metric="euclidean", random_state=RANDOM_SEED)
umap_op.fit(adata_merged.obsm["X_scANVI"])
joblib.dump(umap_op, output_dir / f"{OUTPUT_PREFIX}_umap_scanvi_operator.joblib")
print("  -> UMAP operator saved")



[Step 16] Computing UMAPs...

[Leiden] Query-only clustering (resolution=1.0)...
  -> 17 query-only clusters
  -> X_umap_scVI saved
  -> X_umap_scANVI saved (default X_umap)
  -> UMAP operator saved


## Cell 20 — Step 16.5: Run Log

In [23]:
print("\n" + "=" * 80)
print("[Step 16.5] Writing run log...")
print("=" * 80)
log_ts   = datetime.now().isoformat()
out_h5ad = output_dir / f"{OUTPUT_PREFIX}_results.h5ad"

pipeline_log = {
    "version": "2.0", "timestamp": log_ts,
    "reference_h5ad": REFERENCE_H5AD, "query_h5ad": QUERY_H5AD,
    "output_dir": str(output_dir), "output_prefix": OUTPUT_PREFIX,
    "planned_output_h5ad": str(out_h5ad),
    "elapsed_min": round((time.time() - PIPELINE_START) / 60, 2),
    "n_obs": int(adata_merged.n_obs), "n_vars": int(adata_merged.n_vars),
    "n_hvg": n_hvg_final, "gpu_available": bool(gpu_available),
    "architecture": "CellTypist-equal scANVI (no ref/query asymmetry in labels)",
    "scanvi_label_source": label_source_col,
}
anndata_structure = {
    "timestamp": log_ts, "shape": [int(adata_merged.n_obs), int(adata_merged.n_vars)],
    "raw": {"present": adata_merged.raw is not None,
            "shape": list(adata_merged.raw.shape) if adata_merged.raw else None},
    "obs_columns": list(adata_merged.obs.columns),
    "obsm_keys":   sorted(adata_merged.obsm.keys()),
    "uns_keys":    sorted(str(k) for k in adata_merged.uns.keys()),
}
adata_merged.uns["pipeline_log"]      = pipeline_log
adata_merged.uns["anndata_structure"] = anndata_structure
(output_dir / f"{OUTPUT_PREFIX}_run_log.txt").write_text(
    "\n".join(f"{k}: {v}" for k,v in pipeline_log.items()), encoding="utf-8"
)
with open(output_dir / f"{OUTPUT_PREFIX}_anndata_structure.json","w") as f:
    json.dump(anndata_structure, f, indent=2)
print("  -> Log written")



[Step 16.5] Writing run log...
  -> Log written


## Cell 21 — Step 17: Save

In [24]:
print("\n" + "=" * 80)
print("[Step 17] Saving...")
print("=" * 80)
_cat_cols = [
    "data_source", "cell_type_fine_ref",
    "scanvi_labels", "scanvi_pred",
    "cd4_cd8_by_score", "leiden_query",
    CELLTYPIST_DIRECT_LABEL_KEY, CELLTYPIST_DIRECT_FILT_KEY,
]
for col in _cat_cols:
    for ad in [adata_merged, adata_train]:
        if col in ad.obs.columns:
            ad.obs[col] = ad.obs[col].astype("category")

scanvi_model.save(output_dir / f"{OUTPUT_PREFIX}_scanvi_model", overwrite=True)
scvi_model.save(output_dir  / f"{OUTPUT_PREFIX}_scvi_model",   overwrite=True)

config = {
    "version":    "2.0",
    "timestamp":  datetime.now().isoformat(),
    "input":      {"reference": REFERENCE_H5AD, "query": QUERY_H5AD},
    "n_hvg":      n_hvg_final,
    "architecture": {
        "description":    "CellTypist-equal scANVI: all cells labeled by CellTypist",
        "scanvi_labels":  label_source_col,
        "ref_labels_col": "cell_type_fine_ref (QC only, not used for training)",
    },
    "annotation_layers": {
        "celltypist_direct": {
            "label_key":      CELLTYPIST_DIRECT_LABEL_KEY,
            "filtered_key":   CELLTYPIST_DIRECT_FILT_KEY,
            "proba_key":      "celltypist_proba",
            "conf_threshold": CELLTYPIST_CONF_THRESHOLD,
        },
        "scanvi": {
            "pred_key":   "scanvi_pred",
            "conf_key":   "scanvi_confidence",
            "latent_key": "X_scANVI",
            "proba_key":  "scanvi_proba",
        },
    },
    "umap_spaces": {
        "X_umap":        "DEFAULT (scANVI-based)",
        "X_umap_scVI":   "scVI latent UMAP",
        "X_umap_scANVI": "scANVI latent UMAP",
    },
    "scanvi_labels": list(label_order),
}
with open(output_dir / f"{OUTPUT_PREFIX}_config.json","w") as f:
    json.dump(config, f, indent=2)


# Annotation statistics CSV (mirrors epithelial pipeline output format)
_stats = (
    adata_merged.obs["scanvi_pred"].value_counts()
    .rename_axis("Cell_Type").reset_index(name="Count")
)
_stats["Percentage"] = 100 * _stats["Count"] / _stats["Count"].sum()
_conf = adata_merged.obs.groupby("scanvi_pred")["scanvi_confidence"].agg(["mean", "std"])
_stats = _stats.merge(_conf, left_on="Cell_Type", right_index=True, how="left")
_stats.to_csv(output_dir / f"{OUTPUT_PREFIX}_scanvi_statistics.csv", index=False)

# Per data_source breakdown
adata_merged.obs[["data_source", "scanvi_pred",
                   CELLTYPIST_DIRECT_LABEL_KEY, CELLTYPIST_DIRECT_FILT_KEY,
                   "scanvi_confidence", "novelty_score"]].to_csv(
    output_dir / f"{OUTPUT_PREFIX}_annotations.csv"
)
print(f"[OK] Annotation CSVs saved")
adata_merged.write_h5ad(out_h5ad, compression="gzip")
print(f"  -> {out_h5ad}")
adata_train.write_h5ad(output_dir / f"{OUTPUT_PREFIX}_train_HVG.h5ad", compression="gzip")
print("  -> train HVG saved")



[Step 17] Saving...
[OK] Annotation CSVs saved
  -> /home/h2048/data/py/20260310/tcell_only_merged_pipeline_v2/tcell_merged_v2_results.h5ad
  -> train HVG saved


## Cell 22 — Step 18: Visualization

In [26]:
print("\n" + "=" * 80)
print("[Step 18] Visualization...")
elapsed = (time.time() - PIPELINE_START) / 60
print("=" * 80)
print(f"PIPELINE COMPLETE  |  Elapsed: {elapsed:.1f} min")

sc.settings.vector_friendly = True

qmask = adata_merged.obs["data_source"] == "query"

# ── Panel 1: Overview 4x4 ────────────────────────────────────────────────
fig = plt.figure(figsize=(24, 20))
gs  = fig.add_gridspec(4, 4, hspace=0.3, wspace=0.3)

panels_row0 = [
    ("data_source",               "Data Source"),
    ("cell_type_fine_ref",        "Ref Original Labels (QC only)"),
    (CELLTYPIST_DIRECT_LABEL_KEY, "CellTypist Direct"),
    (CELLTYPIST_DIRECT_FILT_KEY,  f"CellTypist Filtered (conf>={CELLTYPIST_CONF_THRESHOLD})"),
]
panels_row1 = [
    ("scanvi_pred",       "scANVI Predictions",    None,      None),
    ("scanvi_confidence", "scANVI Confidence",     "viridis", (0, 1)),
    ("novelty_score",     "Novelty Score",         "hot",     (0, 1)),
    ("cd4_cd8_by_score",  "CD4/CD8 by Score (QC)", None,      None),
]

for col_i, (key, title) in enumerate(panels_row0):
    ax = fig.add_subplot(gs[0, col_i])
    if key in adata_merged.obs.columns:
        sc.pl.umap(adata_merged, color=key, ax=ax, show=False,
                   title=title, s=15, legend_loc="on data")

for col_i, (key, title, cmap, vlim) in enumerate(panels_row1):
    ax = fig.add_subplot(gs[1, col_i])
    kw = dict(ax=ax, show=False, title=title, s=15)
    if cmap: kw["cmap"] = cmap
    if vlim: kw["vmin"], kw["vmax"] = vlim
    if key not in adata_merged.obs.columns: continue
    if not cmap:
        kw["legend_loc"] = "on data"
    sc.pl.umap(adata_merged, color=key, **kw)

marker_panels = [
    ("CD3D",  "CD3D (pan-T)"),   ("CD4",    "CD4"),
    ("CD8A",  "CD8A"),           ("GNLY",   "GNLY (NK)"),
    ("FOXP3", "FOXP3 (Treg)"),   ("HAVCR2", "HAVCR2 (Exhaustion)"),
    ("MKI67", "MKI67 (Prolif)"), ("ITGAE",  "ITGAE (TRM)"),
]
for i, (gene, title) in enumerate(marker_panels):
    row_i, col_i = 2 + i // 4, i % 4
    ax = fig.add_subplot(gs[row_i, col_i])
    if adata_merged.raw is not None and gene in adata_merged.raw.var_names:
        sc.pl.umap(adata_merged, color=gene, ax=ax, show=False,
                   title=title, cmap="Reds", s=15, use_raw=True)
    else:
        ax.set_title(f"{title} (not found)")

plt.savefig(output_dir / f"{OUTPUT_PREFIX}_overview.pdf", dpi=300, bbox_inches="tight")
plt.close()
print(f"  -> {OUTPUT_PREFIX}_overview.pdf")

# ── Panel 2: CellTypist vs scANVI ────────────────────────────────────────
compare_keys = [
    (CELLTYPIST_DIRECT_LABEL_KEY, "CellTypist Direct"),
    (CELLTYPIST_DIRECT_FILT_KEY,  "CellTypist Filtered"),
    ("scanvi_pred",               "scANVI (trained on CellTypist labels)"),
]
fig2, axes2 = plt.subplots(1, 3, figsize=(21, 7))
for ax, (key, title) in zip(axes2, compare_keys):
    if key in adata_merged.obs.columns:
        sc.pl.umap(adata_merged, color=key, ax=ax, show=False,
                   title=title, legend_loc="on data", s=10)
plt.tight_layout()
plt.savefig(output_dir / f"{OUTPUT_PREFIX}_celltypist_vs_scanvi.pdf", dpi=300, bbox_inches="tight")
plt.close()
print(f"  -> {OUTPUT_PREFIX}_celltypist_vs_scanvi.pdf")

# ── Panel 3: scVI vs scANVI UMAP ─────────────────────────────────────────
fig3, axes3 = plt.subplots(2, 3, figsize=(18, 12))
for row_i, (basis, label) in enumerate([("X_umap_scVI", "scVI"), ("X_umap_scANVI", "scANVI")]):
    sc.pl.embedding(adata_merged, basis=basis, color="data_source",
                    ax=axes3[row_i, 0], show=False,
                    title=f"Data Source ({label})", s=10)
    sc.pl.embedding(adata_merged, basis=basis, color=CELLTYPIST_DIRECT_LABEL_KEY,
                    ax=axes3[row_i, 1], show=False,
                    title=f"CellTypist ({label})", legend_loc="on data", s=10)
    sc.pl.embedding(adata_merged, basis=basis, color="scanvi_pred",
                    ax=axes3[row_i, 2], show=False,
                    title=f"scANVI ({label})", legend_loc="on data", s=10)
plt.tight_layout()
plt.savefig(output_dir / f"{OUTPUT_PREFIX}_umap_comparison.pdf", dpi=300, bbox_inches="tight")
plt.close()
print(f"  -> {OUTPUT_PREFIX}_umap_comparison.pdf")

# ── Summary ───────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("T/NK PIPELINE v2.1 COMPLETE")
print("=" * 80)
print(f"  Total cells: {adata_merged.n_obs:,}  "
      f"(ref: {(adata_merged.obs['data_source']=='reference').sum():,}  "
      f"query: {(adata_merged.obs['data_source']=='query').sum():,})")
print(f"\nscANVI label source: {label_source_col}")
n_unk = (adata_merged.obs.get("scanvi_labels", pd.Series(dtype=str)) == UNLABELED_CATEGORY).sum()
print(f"  Unknown (low-conf): {n_unk:,}")
print("\nTop scANVI predictions:")
print(adata_merged.obs["scanvi_pred"].value_counts().head(10))
print("\nTop CellTypist direct labels:")
print(adata_merged.obs[CELLTYPIST_DIRECT_LABEL_KEY].value_counts().head(10))
print("=" * 80)


[Step 18] Visualization...
PIPELINE COMPLETE  |  Elapsed: 726.5 min
  -> tcell_merged_v2_overview.pdf
  -> tcell_merged_v2_celltypist_vs_scanvi.pdf
  -> tcell_merged_v2_umap_comparison.pdf

T/NK PIPELINE v2.1 COMPLETE
  Total cells: 190,168  (ref: 44,833  query: 145,335)

scANVI label source: celltypist_label_direct_filt
  Unknown (low-conf): 0

Top scANVI predictions:
scanvi_pred
Regulatory T cells             74221
Trm cytotoxic T cells          35372
Tem/Trm cytotoxic T cells      29623
CD16+ NK cells                 20836
Tem/Temra cytotoxic T cells    16004
CD16- NK cells                 14112
Name: count, dtype: int64

Top CellTypist direct labels:
celltypist_label_direct
Trm cytotoxic T cells          45020
Tem/Trm cytotoxic T cells      36695
Regulatory T cells             25471
Tem/Effector helper T cells    18439
CD16+ NK cells                 18372
Tem/Temra cytotoxic T cells    12943
CD16- NK cells                  8216
Tcm/Naive helper T cells        7059
NK cells        